####　大まかな部分はよさそう。11は大体細街路になっているので主要道のみを抜き出せる。問題としては1123（北にある高砂道路）の端点がなくなっていたこと。こればっかりは回したのちに人力で行うか，作業前に1123につながる部分を2車線にしておくか（現実道路は1車線なのでよろしくない。）

In [10]:
# ファイル読み込み
network_file = r"\\wsl.localhost\ubuntu-22.04\home\tsato-cnlab\Emates\eMATES_2308\network\simple_shikata\network.txt"
position_file = r"\\wsl.localhost\ubuntu-22.04\home\tsato-cnlab\Emates\eMATES_2308\network\simple_shikata\mapPosition.txt"

"""network.txtファイルを解析"""
connections = {}
with open(network_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('//'):
            continue
        
        parts = line.split(',')
        if len(parts) < 2:
            continue
        
        try:
            node_id = int(parts[0])
            lane_config = parts[1]
            connected_nodes = [int(x) for x in parts[2:] if x.strip()]
            
            connections[node_id] = {
                'lane_config': lane_config,
                'connected_nodes': connected_nodes
            }
        except ValueError as e:
            print(f"行の解析エラー: {line} - エラー: {e}")

In [11]:
# ファイル読み込み
network_file = r"\\wsl.localhost\ubuntu-22.04\home\tsato-cnlab\Emates\eMATES_2308\network\simple_shikata\network.txt"
position_file = r"\\wsl.localhost\ubuntu-22.04\home\tsato-cnlab\Emates\eMATES_2308\network\simple_shikata\mapPosition.txt"

# 車線構成を'11'単位で分析し、対応する接続を削除
def analyze_lane_units(lane_config):
    """車線構成を'11'単位で分割"""
    units = []
    i = 0
    while i < len(lane_config):
        if i + 1 < len(lane_config):
            unit = lane_config[i:i+2]
            units.append(unit)
            i += 2
        else:
            # 奇数文字の場合、最後の1文字を単独処理
            units.append(lane_config[i])
            i += 1
    return units

# 新しいフィルタリング処理
filtered_connections_new = {}

for node_id, data in connections.items():
    lane_config = data['lane_config']
    connected_nodes = data['connected_nodes']
    
    # ノードIDが5000以上の場合はそのまま保持
    if node_id >= 5000:
        filtered_connections_new[node_id] = {
            'lane_config': lane_config,
            'connected_nodes': connected_nodes
        }
        continue
    
    # ノードIDが5000未満の場合のみ'11'単位の削除処理
    # 車線構成を'11'単位で分割
    lane_units = analyze_lane_units(lane_config)
    
    # '11'単位のインデックスを特定
    keep_indices = []
    for i, unit in enumerate(lane_units):
        if unit != '11':
            keep_indices.append(i)
    
    # 保持する接続ノードを選択
    filtered_connected = []
    for i in keep_indices:
        if i < len(connected_nodes):
            conn_node = connected_nodes[i]
            # 接続先ノードも存在確認
            if conn_node in connections:
                filtered_connected.append(conn_node)
    
    # 新しい車線構成を作成
    new_lane_config = ''.join([lane_units[i] for i in keep_indices])
    
    # 接続があるノードのみ保持
    if filtered_connected and new_lane_config:
        filtered_connections_new[node_id] = {
            'lane_config': new_lane_config,
            'connected_nodes': filtered_connected
        }

print(f"新しいフィルタリング結果: {len(filtered_connections_new)}ノード")

# 変更の詳細表示
print("\n=== 変更詳細 ===")
processed_count = 0
for node_id in sorted(connections.keys()):
    if processed_count >= 10:  # 最初の10ノードのみ表示
        break
        
    original = connections[node_id]
    original_units = analyze_lane_units(original['lane_config'])
    
    print(f"\nノード {node_id}:")
    print(f"  元の車線構成: {original['lane_config']} -> 単位: {original_units}")
    print(f"  元の接続: {original['connected_nodes']}")
    
    if node_id >= 5000:
        print(f"  -> ID>=5000のため保持")
    elif node_id in filtered_connections_new:
        new_data = filtered_connections_new[node_id]
        new_units = analyze_lane_units(new_data['lane_config'])
        print(f"  新しい車線構成: {new_data['lane_config']} -> 単位: {new_units}")
        print(f"  新しい接続: {new_data['connected_nodes']}")
    else:
        print(f"  -> 削除されました")
    
    processed_count += 1

# 統計情報
high_id_nodes = [node_id for node_id in connections.keys() if node_id >= 5000]
low_id_nodes = [node_id for node_id in connections.keys() if node_id < 5000]

original_total_connections = sum(len(data['connected_nodes']) for data in connections.values())
new_total_connections = sum(len(data['connected_nodes']) for data in filtered_connections_new.values())

print(f"\n=== 統計 ===")
print(f"元のノード数: {len(connections)}")
print(f"  ID>=5000のノード: {len(high_id_nodes)}")
print(f"  ID<5000のノード: {len(low_id_nodes)}")
print(f"新しいノード数: {len(filtered_connections_new)}")
print(f"削除されたノード数: {len(connections) - len(filtered_connections_new)}")
print(f"元の総接続数: {original_total_connections}")
print(f"新しい総接続数: {new_total_connections}")
print(f"削除された接続数: {original_total_connections - new_total_connections}")

# 新しいファイルに出力
with open("network_filtered_units.txt", 'w', encoding='utf-8') as f:
    f.write("#+ Filtered network.txt - removed '11' units for nodes with ID<5000\n")
    for node_id in sorted(filtered_connections_new.keys()):
        data = filtered_connections_new[node_id]
        connected_str = ','.join(map(str, data['connected_nodes']))
        f.write(f"{node_id},{data['lane_config']},{connected_str}\n")

print("\n出力完了: network_filtered_units.txt")

# 削除されたノードIDを表示
deleted_nodes = set(connections.keys()) - set(filtered_connections_new.keys())
print(f"\n削除されたノード数: {len(deleted_nodes)}")
print(f"削除されたノードID: {sorted(deleted_nodes)}")

新しいフィルタリング結果: 131ノード

=== 変更詳細 ===

ノード 0:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1437, 1446]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1437, 1446]

ノード 4:
  元の車線構成: 22 -> 単位: ['22']
  元の接続: [5008]
  新しい車線構成: 22 -> 単位: ['22']
  新しい接続: [5008]

ノード 17:
  元の車線構成: 222222 -> 単位: ['22', '22', '22']
  元の接続: [126, 5006, 179]
  新しい車線構成: 222222 -> 単位: ['22', '22', '22']
  新しい接続: [126, 5006, 179]

ノード 32:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1064, 81]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1064, 81]

ノード 57:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1365, 81]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1365, 81]

ノード 81:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [57, 32]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [57, 32]

ノード 126:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1045, 17]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1045, 17]

ノード 169:
  元の車線構成: 22 -> 単位: ['22']
  元の接続: [175]
  新しい車線構成: 22 -> 単位: ['22']
  新しい接続: [175]

ノード 175:
  元の車線構成: 2222 -> 単位: ['2

In [12]:
deleted_nodes_list = sorted(list(deleted_nodes))
print()

with open(position_file, 'r', encoding='utf-8') as f:
    positions = {}
    for line in f:
        line = line.strip()
        if not line or line.startswith('//'):
            continue
        
        parts = line.split(',')
        if len(parts) >= 4:
            node_id = int(parts[0])
            x = float(parts[1])
            y = float(parts[2])
            z = int(parts[3])
            
            positions[node_id] = {'x': x, 'y': y, 'z': z}
print(f"\n元の位置ノード数: {positions.keys()}")
# 削除されたノードをmapPosition.txtでも削除
type(positions.keys())
for node_id in deleted_nodes_list:
    if node_id in positions:
        del positions[node_id]
print(f"削除された位置ノード数: {len(positions)}")
with open("mapPosition_filtered_units.txt", 'w', encoding='utf-8') as f:
    f.write("#+ Filtered mapPosition.txt - removed positions for deleted nodes\n")
    for node_id, pos in sorted(positions.items()):
        f.write(f"{node_id},{pos['x']},{pos['y']},{pos['z']}\n")
    print("出力完了: mapPosition_filtered_units.txt")



元の位置ノード数: dict_keys([0, 4, 17, 32, 57, 81, 126, 169, 175, 179, 194, 204, 210, 214, 244, 245, 246, 248, 249, 253, 254, 255, 256, 257, 275, 328, 402, 414, 415, 457, 464, 466, 475, 522, 525, 551, 613, 623, 624, 651, 660, 666, 668, 683, 713, 716, 739, 826, 885, 887, 892, 934, 956, 1005, 1010, 1012, 1023, 1031, 1035, 1036, 1037, 1045, 1048, 1049, 1060, 1064, 1070, 1086, 1087, 1088, 1113, 1123, 1196, 1281, 1309, 1347, 1365, 1379, 1384, 1395, 1400, 1401, 1407, 1425, 1437, 1442, 1443, 1445, 1446, 1447, 1448, 1449, 1458, 1459, 1460, 1461, 1468, 1469, 1470, 1471, 3003, 3012, 3030, 3031, 3032, 3033, 3040, 5001, 5003, 5005, 5006, 5008, 5010, 5011, 5013, 900000, 900001, 900002, 900003, 900004, 900005, 900006, 900007, 900008, 900009, 900010, 900011, 900012, 900013, 900014, 900015])
削除された位置ノード数: 131
出力完了: mapPosition_filtered_units.txt


In [13]:
filtered_positions = positions
filtered_connections = filtered_connections_new
print(f"フィルタ後の位置ノード数: {len(filtered_positions)}")
print(f"フィルタ後のネットワークノード数: {len(filtered_connections)}")

フィルタ後の位置ノード数: 131
フィルタ後のネットワークノード数: 131


In [14]:
# 車線構成を'11'単位で分析し、対応する接続を削除
def analyze_lane_units(lane_config):
    """車線構成を'11'単位で分割"""
    units = []
    i = 0
    while i < len(lane_config):
        if i + 1 < len(lane_config):
            unit = lane_config[i:i+2]
            units.append(unit)
            i += 2
        else:
            # 奇数文字の場合、最後の1文字を単独処理
            units.append(lane_config[i])
            i += 1
    return units

# 新しいフィルタリング処理
filtered_connections_new = {}

for node_id, data in connections.items():
    lane_config = data['lane_config']
    connected_nodes = data['connected_nodes']
    
    # 車線構成を'11'単位で分割
    lane_units = analyze_lane_units(lane_config)
    
    # '11'単位のインデックスを特定
    keep_indices = []
    for i, unit in enumerate(lane_units):
        if unit != '11':
            keep_indices.append(i)
    
    # 保持する接続ノードを選択
    filtered_connected = []
    for i in keep_indices:
        if i < len(connected_nodes):
            conn_node = connected_nodes[i]
            # 接続先ノードも存在確認
            if conn_node in connections:
                filtered_connected.append(conn_node)
    
    # 新しい車線構成を作成
    new_lane_config = ''.join([lane_units[i] for i in keep_indices])
    
    # 接続があるノードのみ保持
    if filtered_connected and new_lane_config:
        filtered_connections_new[node_id] = {
            'lane_config': new_lane_config,
            'connected_nodes': filtered_connected
        }

print(f"新しいフィルタリング結果: {len(filtered_connections_new)}ノード")

# 変更の詳細表示
print("\n=== 変更詳細 ===")
for node_id in sorted(list(connections.keys())[:10]):  # 最初の10ノードで例示
    if node_id in connections:
        original = connections[node_id]
        original_units = analyze_lane_units(original['lane_config'])
        
        print(f"\nノード {node_id}:")
        print(f"  元の車線構成: {original['lane_config']} -> 単位: {original_units}")
        print(f"  元の接続: {original['connected_nodes']}")
        
        if node_id in filtered_connections_new:
            new_data = filtered_connections_new[node_id]
            new_units = analyze_lane_units(new_data['lane_config'])
            print(f"  新しい車線構成: {new_data['lane_config']} -> 単位: {new_units}")
            print(f"  新しい接続: {new_data['connected_nodes']}")
        else:
            print(f"  -> 削除されました")

# 統計情報
original_total_connections = sum(len(data['connected_nodes']) for data in connections.values())
new_total_connections = sum(len(data['connected_nodes']) for data in filtered_connections_new.values())

print(f"\n=== 統計 ===")
print(f"元のノード数: {len(connections)}")
print(f"新しいノード数: {len(filtered_connections_new)}")
print(f"削除されたノード数: {len(connections) - len(filtered_connections_new)}")
print(f"元の総接続数: {original_total_connections}")
print(f"新しい総接続数: {new_total_connections}")
print(f"削除された接続数: {original_total_connections - new_total_connections}")

# 新しいファイルに出力
with open("network_filtered_units.txt", 'w', encoding='utf-8') as f:
    f.write("// Filtered network.txt - removed '11' units and corresponding connections\n")
    for node_id in sorted(filtered_connections_new.keys()):
        data = filtered_connections_new[node_id]
        connected_str = ','.join(map(str, data['connected_nodes']))
        f.write(f"{node_id},{data['lane_config']},{connected_str}\n")

print("\n出力完了: network_filtered_units.txt")

新しいフィルタリング結果: 115ノード

=== 変更詳細 ===

ノード 0:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1437, 1446]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1437, 1446]

ノード 4:
  元の車線構成: 22 -> 単位: ['22']
  元の接続: [5008]
  新しい車線構成: 22 -> 単位: ['22']
  新しい接続: [5008]

ノード 17:
  元の車線構成: 222222 -> 単位: ['22', '22', '22']
  元の接続: [126, 5006, 179]
  新しい車線構成: 222222 -> 単位: ['22', '22', '22']
  新しい接続: [126, 5006, 179]

ノード 32:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1064, 81]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1064, 81]

ノード 57:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1365, 81]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1365, 81]

ノード 81:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [57, 32]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [57, 32]

ノード 126:
  元の車線構成: 2222 -> 単位: ['22', '22']
  元の接続: [1045, 17]
  新しい車線構成: 2222 -> 単位: ['22', '22']
  新しい接続: [1045, 17]

ノード 169:
  元の車線構成: 22 -> 単位: ['22']
  元の接続: [175]
  新しい車線構成: 22 -> 単位: ['22']
  新しい接続: [175]

ノード 175:
  元の車線構成: 2222 -> 単位: ['2